# Statistician Agent with Strands
This notebook demonstrates how to build a statistician agent using the open-source [Strands Agents](https://strandsagents.com/latest/) framework and how to deploy the agent on [Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/).

The agent uses three tools powered by the **AgentCore CodeInterpreter** sandbox for code execution:

- `run_code`: Executes agent-generated Python code in the sandbox (used for bar charts and general visualizations)
- `plot_kaplan_meier`: Generates Kaplan-Meier survival plots using lifelines/plotly in the sandbox
- `fit_survival_regression`: Performs Cox regression analysis using lifelines in the sandbox

#### Install Strands agents and required dependencies
`strands-agents-tools >= 0.2.0` is required for Code Interpreter tool support.

In [ ]:
%pip install strands-agents "strands-agents-tools>=0.2.0" bedrock-agentcore matplotlib Pillow --quiet

#### Ensure the latest version of boto3 is shown below
Ensure the boto3 version printed below is **1.39** or higher.

In [ ]:
%pip show boto3

#### Import required libraries

In [ ]:
import boto3
import json
import uuid
from typing import List
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools.code_interpreter import AgentCoreCodeInterpreter
from strands_tools.code_interpreter.models import ExecuteCodeAction, ExecuteCommandAction
from bedrock_agentcore.tools.code_interpreter_client import CodeInterpreter as CodeInterpreterClient
from bedrock_agentcore.memory.client import MemoryClient
from utils.boto3_helper import find_s3_bucket_name_by_suffix, get_role_arn

# Get AWS account information
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']
region = boto3.Session().region_name

# Define Bedrock model id
MODEL_ID = "global.anthropic.claude-sonnet-4-20250514-v1:0"

# Initialize AgentCore Memory for sharing query results between agents
memory_client = MemoryClient(region_name=region)
memory_resource = memory_client.create_or_get_memory(
    name="biomarker_agent_memory",
    strategies=[],
    description="Short-term memory for sharing query results between agents",
    event_expiry_days=3
)
memory_id = memory_resource.get("memoryId") or memory_resource.get("id")
memory_session_id = str(uuid.uuid4())
print(f"Memory ID: {memory_id}, Session ID: {memory_session_id}")

#### Cross-agent data sharing with AgentCore Memory

When running as part of the multi-agent system (notebook 05), the **biomarker database analyst** agent queries Redshift and stores raw query results in [AgentCore Memory](https://docs.aws.amazon.com/bedrock/latest/userguide/agentcore-memory.html) — a short-term, session-scoped storage. The **statistician agent** then retrieves those results directly from memory when performing survival regression analysis, eliminating the need for intermediate S3 storage.

This approach has two key benefits:
- **No manual data passing** — the `fit_survival_regression` tool automatically retrieves the latest query results from memory without requiring S3 bucket or key parameters.
- **Automatic cleanup** — memory events expire after 3 days, so there is no need to manage temporary files.

## Prerequisites

Run through the notebook environment setup in [00-setup_environment.ipynb](00-setup_environment.ipynb).

#### Setup custom CodeInterpreter with execution role and S3 configuration

To allow code running inside the AgentCore CodeInterpreter sandbox to access AWS services (e.g., writing files to S3), we create a **custom code interpreter** with an IAM execution role. The role's trust policy must allow `bedrock-agentcore.amazonaws.com` to assume it, and it must have the necessary permissions (e.g., S3 access). The `BedrockAgentCoreStrands` role provisioned by the infrastructure stack already satisfies these requirements.

> **Important:** Your SageMaker execution role (or local IAM role) must have permissions to call AgentCore APIs (`bedrock-agentcore:CreateCodeInterpreter`, `bedrock-agentcore:StartCodeInterpreterSession`, etc.) and `iam:PassRole` for the AgentCore execution role. See the [README](README.md#prerequisites) for the full list of required permissions.

In [ ]:
import time

# Retrieve bucket information
s3_bucket = find_s3_bucket_name_by_suffix('-agent-build-bucket')
if not s3_bucket:
    print("Error: S3 bucket with suffix '-agent-build-bucket' not found!")

# Retrieve the IAM execution role for the code interpreter sandbox
execution_role_arn = get_role_arn('BedrockAgentCoreStrands')
print(f"Execution Role ARN: {execution_role_arn}")

# Create a custom code interpreter with the execution role
ci_client = CodeInterpreterClient(region)
interpreter_name = "statistician_interpreter"

try:
    ci_response = ci_client.create_code_interpreter(
        name=interpreter_name,
        execution_role_arn=execution_role_arn,
        description="Code interpreter with S3 access for statistician agent",
        network_configuration={"networkMode": "PUBLIC"}
    )
    custom_identifier = ci_response["codeInterpreterId"]
    print(f"Created custom code interpreter: {custom_identifier}")

    # Wait for the interpreter to become READY
    print("Waiting for code interpreter to be ready...")
    while True:
        status_response = ci_client.get_code_interpreter(interpreter_id=custom_identifier)
        status = status_response.get("status", "UNKNOWN")
        if status == "READY":
            print("Code interpreter is READY.")
            break
        elif status in ("FAILED", "DELETING"):
            raise RuntimeError(f"Code interpreter entered unexpected status: {status}")
        time.sleep(5)

except ci_client.control_plane_client.exceptions.ConflictException:
    # Interpreter already exists, retrieve its ID
    interpreters = ci_client.list_code_interpreters()
    custom_identifier = None
    for ci in interpreters.get("codeInterpreterSummaries", []):
        if ci["name"] == interpreter_name:
            custom_identifier = ci["codeInterpreterId"]
            break
    if not custom_identifier:
        raise RuntimeError(f"Code interpreter '{interpreter_name}' conflict but could not find existing one")
    print(f"Using existing custom code interpreter: {custom_identifier}")

# Initialize AgentCore CodeInterpreter with the custom identifier
code_interpreter = AgentCoreCodeInterpreter(
    region=region,
    identifier=custom_identifier,
    session_name="statistician-session"
)

# One-time sandbox setup: install required packages
_sandbox_initialized = False

def ensure_sandbox():
    """Install required packages in the CodeInterpreter sandbox on first use."""
    global _sandbox_initialized
    if not _sandbox_initialized:
        print("Installing packages in CodeInterpreter sandbox...")
        code_interpreter.execute_command(
            ExecuteCommandAction(
                type="executeCommand",
                command="pip install lifelines boto3 pandas numpy matplotlib"
            )
        )
        _sandbox_initialized = True
        print("Sandbox packages installed.")

print(f"Region: {region}")
print(f"Account ID: {account_id}")
print(f"S3 bucket: {s3_bucket}")
print(f"Code Interpreter ID: {custom_identifier}")

## Strands Agent Creation
In this section we create the statistician agent using the Strands framework

#### Define agent configuration and instructions

In [ ]:
statistician_agent_name = 'Statistician-strands'
statistician_agent_description = "scientific analysis for survival analysis using Strands framework"
statistician_agent_instruction = f"""You are a medical research assistant AI specialized in survival analysis with biomarkers. 
Your primary job is to interpret user queries, run scientific analysis tasks, and provide relevant medical insights 
with available visualization tools. Use only the appropriate tools as required by the specific question. 
Follow these instructions carefully: 

1. If the user query requires a Kaplan-Meier chart: 
   a. Map survival status as 0 for Alive and 1 for Dead for the event parameter. 
   b. Use survival duration as the duration parameter. 
   c. Use the group_survival_data tool to create baseline and condition group based on expression value threshold provided by the user. 

2. If a survival regression analysis is needed: 
   a. You need access to all records with columns start with survival status as first column, then survival duration, and the required biomarkers. 
   b. Use the fit_survival_regression tool to identify the best-performing biomarker based on the p-value summary. 
   c. The tool automatically retrieves the latest query results from shared memory. No S3 path is needed. 

3. When you need to create a bar chart or any visualization not covered by the specialized tools:
   a. Use the run_code tool to write and execute Python code in the sandbox.
   b. Use matplotlib to create the chart and save the image to S3.
   c. The S3 bucket is: {s3_bucket}
   d. Save charts under the 'graphs/' prefix in the bucket.
   e. Use 'Agg' backend for matplotlib (matplotlib.use('Agg')).
   f. Use boto3 to upload the image to S3.

4. When providing your response: 
   a. Start with a brief summary of your understanding of the user's query. 
   b. Explain the steps you're taking to address the query. 
   Ask for clarifications from the user if required. 
   c. If you generate any charts or perform statistical analyses, 
   explain their significance in the context of the user's query. 
   d. Conclude with a concise summary of the findings and their potential implications for medical research. 
   e. Make sure to explain any medical or statistical concepts in a clear, accessible manner.
"""

#### Define tools for Strands agent
The `run_code` tool lets the agent generate and execute its own Python code in the CodeInterpreter sandbox (e.g., for bar charts). The other tools use predefined code templates for specialized statistical analyses.

In [ ]:
@tool
def run_code(code: str) -> str:
    """
    Execute Python code in the CodeInterpreter sandbox.
    Use this tool to write and run any Python code, including creating
    charts, performing calculations, or processing data.

    The sandbox has matplotlib, pandas, numpy, lifelines, and boto3 available.
    To save charts to S3, use boto3 to upload to the bucket and prefix shown below.

    S3 bucket: {s3_bucket}
    S3 prefix: graphs/

    Args:
        code (str): Python code to execute in the sandbox

    Returns:
        str: Output from the code execution
    """
    ensure_sandbox()

    print(f"\nExecuting code in sandbox...\n")
    result = code_interpreter.execute_code(
        ExecuteCodeAction(type="executeCode", code=code, language="python")
    )
    print(f"\nExecution result: {json.dumps(result, indent=2)}\n")
    return json.dumps(result, indent=2)


@tool
def plot_kaplan_meier(biomarker_name: str, duration_baseline: List[float], duration_condition: List[float],
                     event_baseline: List[int], event_condition: List[int]) -> str:
    """
    Create a Kaplan-Meier survival plot for comparing two groups.

    Args:
        biomarker_name (str): Name of the biomarker being analyzed
        duration_baseline (List[float]): Survival duration in days for baseline group
        duration_condition (List[float]): Survival duration in days for condition group
        event_baseline (List[int]): Survival events for baseline (0=alive, 1=dead)
        event_condition (List[int]): Survival events for condition (0=alive, 1=dead)

    Returns:
        str: Result of the Kaplan-Meier plot creation
    """
    ensure_sandbox()

    code = f"""
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
import io
import boto3

biomarker_name = {repr(biomarker_name)}
duration_baseline = {duration_baseline}
event_baseline = {event_baseline}
duration_condition = {duration_condition}
event_condition = {event_condition}
s3_bucket = {repr(s3_bucket)}

baseline_label = '<=10'
condition_label = '>10'

kmf_baseline = KaplanMeierFitter()
kmf_baseline.fit(durations=duration_baseline, event_observed=event_baseline, label=baseline_label)

kmf_condition = KaplanMeierFitter()
kmf_condition.fit(durations=duration_condition, event_observed=event_condition, label=condition_label)

fig, ax = plt.subplots(figsize=(10, 6))
kmf_baseline.plot_survival_function(ax=ax, ci_show=True, color='blue')
kmf_condition.plot_survival_function(ax=ax, ci_show=True, color='darkorange')
ax.set_title(biomarker_name)
ax.set_xlabel('Timeline (days)')
ax.set_ylabel('Survival Probability')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

img_data = io.BytesIO()
fig.savefig(img_data, format='png', dpi=150, bbox_inches='tight')
img_data.seek(0)

s3 = boto3.resource('s3')
key = f'graphs/{{biomarker_name}}_KMplot.png'
s3.Bucket(s3_bucket).put_object(Body=img_data, ContentType='image/png', Key=key)
print(f"Kaplan-Meier plot saved to s3://{{s3_bucket}}/{{key}}")
"""
    print(f"\nCreating Kaplan-Meier plot for: {biomarker_name}\n")
    result = code_interpreter.execute_code(
        ExecuteCodeAction(type="executeCode", code=code, language="python")
    )
    print(f"\nKaplan-Meier Output: {json.dumps(result, indent=2)}\n")
    return json.dumps(result, indent=2)


@tool
def fit_survival_regression() -> str:
    """
    Fit a survival regression model using query results stored in AgentCore Memory.
    This tool automatically retrieves the latest query results from memory that were
    saved by the biomarker database analyst agent. No parameters are needed.

    Returns:
        str: Results of the survival regression analysis
    """
    ensure_sandbox()

    # Retrieve query results from AgentCore Memory
    events = memory_client.list_events(
        memory_id=memory_id,
        actor_id="biomarker_agent",
        session_id=memory_session_id,
        event_metadata=[{
            "left": {"metadataKey": "type"},
            "operator": "EQUALS_TO",
            "right": {"metadataValue": {"stringValue": "query_results"}}
        }],
        max_results=10,
        include_payload=True
    )

    if not events:
        return json.dumps({"status": "error", "content": [{"text": "No query results found in memory. Please run the biomarker database query first."}]})

    # Get the most recent query results
    latest_event = events[-1]
    data_text = latest_event["payload"][0]["conversational"]["content"]["text"]
    print(f"Retrieved query results from memory (event: {latest_event['eventId']})")

    # Pass the data directly to the sandbox as a JSON string
    import base64
    data_b64 = base64.b64encode(data_text.encode()).decode()

    code = f"""
import json
import base64
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter

# Read data from memory (passed as base64-encoded JSON)
data = json.loads(base64.b64decode("{data_b64}").decode())

# Process clinical genomic data
columns = [col['name'] for col in data['ColumnMetadata']]
processed_records = []
for record in data['Records']:
    row = []
    for value in record:
        if 'stringValue' in value:
            row.append(value['stringValue'])
        elif 'doubleValue' in value:
            row.append(value['doubleValue'])
        elif 'booleanValue' in value:
            row.append(value['booleanValue'])
        else:
            row.append(None)
    processed_records.append(row)

df = pd.DataFrame(processed_records, columns=columns)

# Map survival status
df['survival_status'] = df['survival_status'].map({{False: 0, True: 1}})

# Data adjustments for alive patients
df.loc[df['survival_status'] == 0, 'survival_duration'] = 100
for biomarker in ['gdf15', 'lrig1', 'cdh2', 'postn', 'vcan']:
    if biomarker in df.columns:
        mask = df['survival_status'] == 0
        df.loc[mask, biomarker] = df.loc[mask, biomarker] + (np.random.rand(mask.sum()) * 30)

df_numeric = df.select_dtypes(include='number')

# Fit Cox Proportional Hazards model
cph = CoxPHFitter(penalizer=0.01)
cph.fit(df_numeric, duration_col='survival_duration', event_col='survival_status')
summary = cph.summary

print("Cox Proportional Hazards Regression Summary:")
print(summary.to_string())
"""
    print(f"\nFitting survival regression with data from AgentCore Memory\n")
    result = code_interpreter.execute_code(
        ExecuteCodeAction(type="executeCode", code=code, language="python")
    )
    print(f"\nSurvival Regression Output: {json.dumps(result, indent=2)}\n")
    return json.dumps(result, indent=2)


# Create list of tools
statistician_tools = [run_code, plot_kaplan_meier, fit_survival_regression]
print(f"Created {len(statistician_tools)} tools for the Strands agent")

#### Setup AWS Bedrock provider for Strands

In [ ]:
# Create Bedrock model for Strands
model = BedrockModel(
    model_id=MODEL_ID,
    region_name=region,
    temperature=0.1,
    streaming=True
)

#### Create the Strands agent

In [ ]:
# Create the Strands agent
try:
    statistician_agent = Agent(
        model=model,
        tools=statistician_tools,
        system_prompt=statistician_agent_instruction
    )
    
    print(f"Successfully created Strands agent: {statistician_agent_name}")
    print(f"Agent has {len(statistician_tools)} tools available")
    
except Exception as e:
    print(f"Error creating agent: {e}")
    raise

#### Test the Strands agent locally

In [ ]:
# Test the agent with a bar chart creation
test_query = """Create me a bar chart for the top 5 gene biomarkers (e.g.,TP53, BRCA1, EGFR, KRAS, MYC)
with respect to their prognostic significance in chemotherapy-treated patients.
The Y-axis should represent –log10(p-value) from a Cox proportional hazards model assessing association with overall survival. 
Y-axis values are: 8.3, 6.7, 5.9, 4.2, 3.8
"""

print(f"Testing agent with query: {test_query}")
print("=" * 80)

try:
    # Run the agent
    statistician_agent(test_query)
except Exception as e:
    print(f"Error during agent execution: {e}")
    import traceback
    traceback.print_exc()

#### Accessing the generated charts

The files are stored in the S3 bucket with name containing 'env*-agent-build-bucket' under the **/graphs** folder. You can also run the cell below to visualize the files in the notebook.

In [ ]:
# Visualize the charts generated
from IPython.display import display, Image
from PIL import Image as PILImage

def display_s3_folder_files(bucket_name, folder_prefix=""):
    """Display all files from an S3 folder in a Jupyter notebook"""
    s3_client = boto3.client('s3')
    response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=folder_prefix)
    
    if 'Contents' not in response:
        print(f"No files found in {bucket_name}/{folder_prefix}")
        return
    
    files_data = []
    for obj in response['Contents']:
        key = obj['Key']
        print(f"\n📁 {key} ({obj['Size']} bytes)")
        
        # Check if file is an image
        if key.lower().endswith('png'):
            try:
                # Download and display image
                img_obj = s3_client.get_object(Bucket=bucket_name, Key=key)
                img_data = img_obj['Body'].read()
                display(Image(data=img_data, width=600))
            except Exception as e:
                print(f"❌ Could not display image: {e}")

# Usage example:
display_s3_folder_files(s3_bucket, 'graph')

#### Advanced usage examples

In [ ]:
# Example of more complex statistical analyses
advanced_queries = [
    """Create a Kaplan-Meier survival plot comparing high vs low expression of EGFR biomarker.
    Use the following sample data:
    - Biomarker name: EGFR
    - High expression group (baseline): survival durations in days [365, 420, 500, 180, 600, 720, 300, 450, 650, 380] 
      and survival events [0, 1, 0, 1, 0, 0, 1, 0, 0, 1] where 0=alive, 1=dead
    - Low expression group (condition): survival durations in days [200, 150, 320, 280, 400, 180, 250, 350, 300, 220]
      and survival events [1, 1, 0, 1, 0, 1, 1, 0, 1, 1] where 0=alive, 1=dead""",
    
    """Generate a bar chart showing average survival duration across different cancer stages.
    Create a bar chart with:
    - Title: "Average Survival Duration by Cancer Stage"
    - X-axis label: "Cancer Stage" 
    - X-axis values: ["Stage I", "Stage II", "Stage III", "Stage IV"]
    - Y-axis label: "Average Survival (months)"
    - Y-axis values: [36.5, 28.2, 18.7, 8.4]""",
    
    """Create a Kaplan-Meier plot for TP53 mutation status comparing mutant vs wild-type patients.
    Use this sample data:
    - Biomarker name: TP53_mutation
    - Wild-type group (baseline): survival durations [450, 380, 520, 290, 600, 340, 480, 390, 550, 420]
      and events [0, 1, 0, 1, 0, 1, 0, 0, 0, 1]
    - Mutant group (condition): survival durations [180, 220, 160, 240, 200, 190, 210, 170, 230, 195]
      and events [1, 1, 1, 0, 1, 1, 0, 1, 1, 1]""",
    
    """Generate a bar chart showing biomarker expression levels in smokers vs non-smokers.
    Create a chart with:
    - Title: "Average Biomarker Expression: Smokers vs Non-smokers"
    - X-axis label: "Patient Group"
    - X-axis values: ["Current Smokers", "Former Smokers", "Never Smokers"]
    - Y-axis label: "Average LRIG1 Expression Level"
    - Y-axis values: [15.2, 22.8, 28.4]"""
]

for query in advanced_queries:
    print(f"\nTesting query: {query}")
    print("-" * 80)
    
    try:
        statistician_agent(query)
    except Exception as e:
        print(f"Error: {e}")

#### Session management and conversation continuity

In [ ]:
# Demonstrate conversation continuity
def interactive_session():
    """
    Simple interactive session with the agent
    """
    print("Interactive Statistical Analysis Session")
    print("Type 'quit' to exit")
    print("=" * 50)
    
    while True:
        user_input = input("\nYour question: ")
        
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Session ended.")
            break
            
        try:
            statistician_agent(user_input)
        except Exception as e:
            print(f"Error: {e}")

interactive_session()

## Deploy agent on AgentCore
In this section we are going to deploy the statistician agent to [Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/) Runtime.

### Preparing your agent for deployment on AgentCore Runtime

In [ ]:
%%writefile statistician_agent_runtime.py

import json
import uuid
import strands
from strands import Agent
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from statistician_agent import *

# Create the Strands agent
try:
    agent = Agent(
        model=model,
        tools=statistician_tools,
        system_prompt=statistician_agent_instruction
    )
    
    print("Successfully created Strands agent")
    print(f"Agent has {len(statistician_tools)} tools available")

except Exception as e:
    print(f"Error creating agent: {e}")
    raise

# Custom event serializer function
def json_serializer(obj):
    if isinstance(obj, strands.agent.agent.Agent):
        return obj.name
    elif isinstance(obj, strands.agent.agent_result.AgentResult):
        result = {}
        result["message"] = str(obj)
        result["metrics"] = {}
        result["metrics"]["accumulated_usage"] = obj.metrics.accumulated_usage
        result["metrics"]["accumulated_metrics"] = obj.metrics.accumulated_metrics
        return result
    elif isinstance(obj, uuid.UUID):
        return str(obj)
    else:
        # Assume other objects are not serializable
        return None

app = BedrockAgentCoreApp()

@app.entrypoint
async def strands_agent_bedrock_streaming(payload):
    """
    Invoke the agent with streaming capabilities
    This function demonstrates how to implement streaming responses
    with AgentCore Runtime using async generators
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    
    try:
        # Stream each chunk as it becomes available
        async for event in agent.stream_async(user_input):
            print("Received event:", event)
            # serialize the event object
            yield json.dumps(event, default=json_serializer)
    except Exception as e:
        # Handle errors gracefully in streaming context
        error_response = {"error": str(e), "type": "stream_error"}
        print(f"Streaming error: {error_response}")
        yield error_response

if __name__ == "__main__":
    app.run()

### Deploying the agent to AgentCore Runtime

#### Define agent name and retrieve runtime role

In [ ]:
from utils.boto3_helper import get_role_arn
iam = boto3.client('iam')

agent_name="statistician_agent"
agentcore_iam_role = get_role_arn('BedrockAgentCoreStrands')
agentcore_iam_role

#### Configure AgentCore Runtime deployment
During the configure step, your docker file will be generated based on your application code.

In [ ]:
import os
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

# Remove existing Dockerfile so configure() regenerates it with the correct entrypoint
if os.path.exists("Dockerfile"):
    os.remove("Dockerfile")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="statistician_agent_runtime.py",
    execution_role=agentcore_iam_role,
    auto_create_ecr=True,
    requirements_file="runtime_requirements.txt",
    region=region,
    agent_name=agent_name
)
response

#### Launching agent to AgentCore Runtime
Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

In [ ]:
launch_result = agentcore_runtime.launch(
    auto_update_on_conflict=True
)
launch_result

### Invoking AgentCore Runtime with boto3
Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [ ]:
import json
import boto3
from IPython.display import Markdown, display

agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

test_query = """Create me a bar chart for the top 5 gene biomarkers (e.g.,TP53, BRCA1, EGFR, KRAS, MYC)
with respect to their prognostic significance in chemotherapy-treated patients.
The Y-axis should represent –log10(p-value) from a Cox proportional hazards model assessing association with overall survival. 
Y-axis values are: 8.3, 6.7, 5.9, 4.2, 3.8
"""
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": test_query})
)

print(f"Response contentType: {response.get("contentType")}\n")
print(f"Testing agent: {test_query}")
print("=" * (15 + len(test_query)))

if "text/event-stream" in response.get("contentType", ""):
    # Processing streaming response
    for line in response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                # remove the SSE structure
                data = line[6:]
                # we need to parse it twice to convert from JSON str to a dictionary
                data_obj = json.loads(data)
                data_obj = json.loads(data_obj)
                # for this example we only care about the data field
                if "data" in data_obj:
                    print(data_obj.get("data"), end="", flush=True)
    print()  # final newline after streaming completes
else:
    # Handle non-streaming response
    try:
        events = []
        for event in response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    if events:
        try:
            response_data = json.loads(events[0].decode("utf-8"))
            display(Markdown(response_data))
        except:
            print(f"Raw response: {events[0]}")